# DELTA LAKE

Local instance to manage MatrizActividades

In [1]:
import pandas as pd
import re
import json
import logging
import pymongo
from pymongo.errors import ConnectionFailure

from eerssa.secret import Keys


from deltalake import DeltaTable, write_deltalake
import os
import sys

from pprint import pprint

logging.basicConfig(level=logging.INFO)

Success!!!


In [2]:


# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


## PyMongo - Descargar una / varias JSON_OT

In [3]:
from pprint import pprint
regex_pattern = re.compile("^2025-07")
query_filter = {"fecha": regex_pattern}


document = CurrentCollection.find_one(query_filter)
pprint(document)

{'_id': ObjectId('6868ac76df6ab3710151861c'),
 'accidente': 'NO',
 'actividades': [{'Actividad': 'PROG',
                  'Ali': None,
                  'Alimentador': None,
                  'Evento': 'En la agencia de la EERSSA Zamora se coordina los '
                            'trabajos, se equipa\n'
                            'el vehículo con materiales y herramientas.',
                  'FinEvento': '2025-07-03 08:40:00',
                  'InicioEvento': '2025-07-03 08:00:00',
                  'Item': '1',
                  'Tipo': 'RUTINARIA'},
                 {'Actividad': 'PROG',
                  'Ali': 'ALI',
                  'Alimentador': 'Zamora I',
                  'Evento': 'Zamora, estructura # 36280, se coordina con el '
                            'interesado, para la\n'
                            'instalación de 1 servicio ocasional, para lo cual '
                            'se hase la\n'
                            'inspección, luego en la bodega de la 

In [4]:
type(document)

dict

In [5]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [2]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

## Hidratar de JSON_OT  a  obj_ot

Dado una documento `json_ot` válido, volverlo a convertir en un objeto de la clase `gestionOt`

In [6]:
pprint(document)

{'_id': ObjectId('6868ac76df6ab3710151861c'),
 'accidente': 'NO',
 'actividades': [{'Actividad': 'PROG',
                  'Ali': None,
                  'Alimentador': None,
                  'Evento': 'En la agencia de la EERSSA Zamora se coordina los '
                            'trabajos, se equipa\n'
                            'el vehículo con materiales y herramientas.',
                  'FinEvento': '2025-07-03 08:40:00',
                  'InicioEvento': '2025-07-03 08:00:00',
                  'Item': '1',
                  'Tipo': 'RUTINARIA'},
                 {'Actividad': 'PROG',
                  'Ali': 'ALI',
                  'Alimentador': 'Zamora I',
                  'Evento': 'Zamora, estructura # 36280, se coordina con el '
                            'interesado, para la\n'
                            'instalación de 1 servicio ocasional, para lo cual '
                            'se hase la\n'
                            'inspección, luego en la bodega de la 

In [9]:
ot = OrdenTrabajo.GestionOt.from_dict( document )
type(ot)

eerssa.gestionOT.GestionOt

In [10]:
matriz = Actividades.ConvertirOT_a_ActividadesCSV(ot)
type(matriz)

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


pandas.core.frame.DataFrame

In [11]:
matriz

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,uuid
0,1,informativa,En la agencia de la EERSSA Zamora se coordina ...,PROG,None,No,No,No,RUTINARIA,·,...,2025-07-03 08:40:00,40,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,812737d3-b7c4-4fef-b532-d07f79aad695
1,2,MEDIDORES,"Zamora, estructura # 36280, se coordina con el...",PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-07-03 09:40:00,60,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,b46dc5c2-7c59-4807-9c3d-f8d9f8f9a282
3,4,REDES,"Zamora barrio La Alvernia, se coordina con los...",PROG,Zamora I,No,No,No,PREVENTIVO,·,...,2025-07-03 13:15:00,215,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,e30ebd0e-4aa7-4584-850c-4fbe09bd3060
5,6,lunch,Lunch en Zamora,ALIMEN,None,No,No,No,LUNCH,·,...,2025-07-03 14:15:00,60,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,b1acff1c-b8a0-4232-a6ad-a3b33b5179fd
6,7,transporte,Traslado desde Zamora hacia Cumbaratza sector ...,TRANSP,None,No,No,No,TRANSPORTE,·,...,2025-07-03 14:36:00,21,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,671f1f1d-5fb0-4e9b-a935-c4e817c2333d
7,8,ALUMBRADO,Cumbaratza sector Rancho Alegre en la estructu...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2025-07-03 14:55:00,19,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,c3f0240b-098e-4d7d-aaa0-6eba08a85e0d
9,10,ALUMBRADO,Cumbaratza sector Rancho Alegre en la estructu...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2025-07-03 15:35:00,40,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,058b0d1b-7ec9-48b2-ad65-e99f3f5fc532
10,11,ALUMBRADO,Cumbaratza sector Rancho Alegre en la estructu...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2025-07-03 15:50:00,15,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,3732d20b-808b-4e3b-9277-e70876b0aa04
11,12,ALUMBRADO,Cumbaratza sector Rancho Alegre en la estructu...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2025-07-03 16:35:00,45,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,1f79dc0c-8edc-4eb8-8096-2d574f6e5d0f
12,13,transporte,Traslado desde Cumbaratza hacia Zamora,TRANSP,None,No,No,No,TRANSPORTE,·,...,2025-07-03 17:05:00,30,MORALES RIVERA LUIS ALBERTO,2,No,2-61,Zamora,156406,sc_pdf_20250704233734_443_pdfreport_ordenesTra...,0266458b-1b03-4a6c-94ab-6ebf53312c39


## DELTA LAKE

### Descargar todo el 2025

Para iniciar crearemos 

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

regex_pattern = re.compile("^2025")
query_filter = {"fecha": regex_pattern}

# Use find() to get a cursor that points to all matching documents
cursor = CurrentCollection.find(query_filter)

obj_list = []
df = pd.DataFrame()
for document in cursor:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  df.join(  )
  print(f"Processing document with id_ot: {document.get('id_ot')}")


In [ ]:
# DELTA_TABLE_PATH_ON_HOST = "/var/lib/docker/volumes/delta_data/_data/my_first_delta_table"
DELTA_TABLE_PATH_ON_HOST = "./delta_data/my_first_delta_table"

In [5]:
df_dummy = pd.DataFrame({"col1": [10, 20], "col2": ["A", "B"]})
write_deltalake(DELTA_TABLE_PATH_ON_HOST, df_dummy)